# Sleep Onset V3 — Full-Recording Training
Standalone controlled experiment. No V2 checkpoint or saved scaler is required.

In [ ]:
from pathlib import Path
import subprocess, sys, zipfile

ON_KAGGLE = Path('/kaggle').exists()
ON_COLAB = 'google.colab' in sys.modules
WORK_DIR = Path('/kaggle/working' if ON_KAGGLE else '/content' if ON_COLAB else Path.cwd())
search_root = Path('/kaggle/input') if ON_KAGGLE else Path.cwd()
candidates = [Path.cwd(), Path.cwd().parent]
candidates += [p.parent for p in search_root.glob('**/config.yaml') if (p.parent / 'train.py').exists()]
PROJECT_DIR = next((p for p in candidates if (p / 'train.py').exists() and (p / 'submission.py').exists() and 'v3_full_recording' in (p / 'config.yaml').read_text()), None)
if PROJECT_DIR is None:
    archives = list(search_root.glob('**/sleep-onset-v3-source.zip'))
    if not archives:
        raise FileNotFoundError('Attach the sleep-onset-v3 source Dataset')
    PROJECT_DIR = WORK_DIR / 'sleep-onset-v3-source'
    with zipfile.ZipFile(archives[0]) as archive:
        archive.extractall(PROJECT_DIR)
DATA_DIR = WORK_DIR / 'neural_compet'
OUTPUT_DIR = WORK_DIR / 'sleep-onset-v3'
print({'project': str(PROJECT_DIR), 'data': str(DATA_DIR), 'output': str(OUTPUT_DIR)})

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_DIR / 'requirements.txt')], check=True)
tests = PROJECT_DIR / 'tests'
if tests.is_dir():
    subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', str(tests)], cwd=PROJECT_DIR, check=True)
else:
    print('Tests were not uploaded; continuing to training.')

## Benchmark and train
Before epoch 1 the script verifies 143,777 parameters, 412/88 train/validation recordings and 61,303 training windows, then benchmarks median, p90 and longest recordings.

In [ ]:
command = [sys.executable, str(PROJECT_DIR / 'train.py'), '--data-dir', str(DATA_DIR), '--output-dir', str(OUTPUT_DIR), '--config', str(PROJECT_DIR / 'config.yaml')]
if not (DATA_DIR / 'Interaxon2026Muse').exists():
    command.append('--download')
subprocess.run(command, cwd=PROJECT_DIR, check=True)

In [ ]:
submission = OUTPUT_DIR / 'submission.zip'
diagnostics = OUTPUT_DIR / 'v3_diagnostics.zip'
with zipfile.ZipFile(submission) as archive:
    assert sorted(archive.namelist()) == ['model_weights.pt', 'submission.py']
    assert 'V3-FullRecording' in archive.read('submission.py').decode()
print((OUTPUT_DIR / 'experiment_summary.md').read_text())
print('Upload only this file to Codabench:', submission)
print('Download for analysis:', diagnostics)